In [ ]:
import openslide
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import glob
import json
import pandas as pd
from tqdm import tqdm
from typing import List
import yaml
import os
from os.path import join as opj
import re
from matplotlib_venn import venn2, venn3
from functools import partial
import matplotlib.patches as mpatches
from itertools import chain

import einops
import cv2
import matplotlib

In [ ]:
def alignment_filter(x):
    if (x["Block"].endswith("x")):
        return False
    
    block_stains = set(x["Stain"])

    if "H&E" not in block_stains:
        return False

    return True

In [ ]:
def get_tumor_label_map(init_pull_fname, btb_pull_fname):
    # get nio SU match meta
    nio_su_match = pd.read_csv(init_pull_fname)
    nio_su_match = nio_su_match[["Pathology Accession #", "tumor type "]].rename(columns={"tumor type ": "tumor", "Pathology Accession #": "su_num"}).drop_duplicates()

    # get btb metadata
    btb_to_pull = pd.read_csv(btb_pull_fname)

    def proc_coarse_labels(x):
        x = json.loads(x.replace("\'", "\""))

        x = set(x)

        if len(x) == 1: return list(x)[0]

        if "todo" in x:
            x.remove("todo")

        if len(x) == 1:
            return list(x)[0]
        else:
            return "todo"

    btb_to_pull["coarse"] = btb_to_pull.drop("Diagnosis 1", axis=1)["coarse"].apply(proc_coarse_labels)
    btb_to_pull = btb_to_pull.drop(["Diagnosis 1"], axis=1).rename({"Accession / ID #": "su_num" , "coarse": "tumor"}, axis=1)

    # aggregating labels, and assert SU-label mapping is unique
    tumor_label_map = pd.concat([btb_to_pull, nio_su_match.dropna()])
    assert (~ tumor_label_map["su_num"].duplicated().any())

    tumor_label_map.loc[tumor_label_map["tumor"]=="schwannoma", "tumor"] = "schwan"
    tumor_label_map.loc[tumor_label_map["tumor"]=="vascular_tumor", "tumor"] = "vascular"
    tumor_label_map.loc[tumor_label_map["tumor"]=="pituitary adenoma", "tumor"] = "pit"
    tumor_label_map.loc[tumor_label_map["tumor"]=="metastatic carcinoma", "tumor"] = "mets"
    tumor_label_map.loc[tumor_label_map["tumor"]=="meningioma", "tumor"] = "mening"
    tumor_label_map.loc[tumor_label_map["tumor"]=="diffuse_hgg", "tumor"] = "glioma"
    tumor_label_map.loc[tumor_label_map["tumor"]=="diffuse_lgg", "tumor"] = "glioma"

    return tumor_label_map

In [ ]:
all_meta = pd.read_csv("/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/master_spreadsheet/matched_su_mxa.csv")
#tumor_label_map = get_tumor_label_map(init_pull_fname="/nfs/mm-isilon/brainscans/dropbox/code/chengjia/to_scan_sorted_nodup_inst.csv", 
#                    btb_pull_fname="/nfs/mm-isilon/brainscans/dropbox/code/chengjia/btb_to_pull.csv")


gpt_labels = pd.read_csv("/nfs/mm-isilon/brainscans/dropbox/code/chengjia/gpt_annot/gpt_annotation_v1v2.csv").drop(["brain_spine_nervous_origin", "tumor_diagnosis", "pathologist", "tumor_origin"], axis=1)
all_meta_with_tumor = all_meta.merge(gpt_labels,left_on="UM Accession", right_on="order", how="left").drop(["order"], axis=1)
all_meta_with_tumor["tumor_coarse_category"] = all_meta_with_tumor["tumor_coarse_category"].fillna("Unknown")
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Diffuse Gliomas", "tumor_coarse_category"] = "glioma"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Sellar Region Tumors", "tumor_coarse_category"] = "sellar"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Meningiomas", "tumor_coarse_category"] = "mening"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Metastatic tumors", "tumor_coarse_category"] = "met"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Circumscribed Astrocytic Gliomas", "tumor_coarse_category"] = "circm_astro"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Glioneuronal and Neuronal Tumors", "tumor_coarse_category"] = "glioneuronal"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Non Neoplastic Brain Conditions", "tumor_coarse_category"] = "not_neo"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Embryonal Tumors", "tumor_coarse_category"] = "embryonal"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Cranial and Paraspinal Nerve Tumors", "tumor_coarse_category"] = "nerve"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Unknown", "tumor_coarse_category"] = "unk"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Mesenchymal Non Meningothelial Tumors", "tumor_coarse_category"] = "mesenchymal"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Ependymal Tumor", "tumor_coarse_category"] = "ependymal"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Non CNS Specimens", "tumor_coarse_category"] = "not_cns"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Hematolymphoid Tumors", "tumor_coarse_category"] = "hematolymphoid"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Chondro-osseous Tumors", "tumor_coarse_category"] = "chondro"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Pineal Tumors", "tumor_coarse_category"] = "pineal"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Choroid Plexus Tumor", "tumor_coarse_category"] = "choroid"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Germ Cell Tumors", "tumor_coarse_category"] = "germ"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Melanocytic Tumors", "tumor_coarse_category"] = "melanocytic"



#all_meta_with_tumor = all_meta.merge(tumor_label_map, left_on="UM Accession", right_on="su_num", how="left").drop("su_num", axis=1)
all_meta_with_tumor["tumor_coarse_category"] = all_meta_with_tumor["tumor_coarse_category"].fillna("unk")
all_meta_with_tumor = all_meta_with_tumor.rename(columns={"tumor_coarse_category":"tumor"})
if all_meta["path"].isna().any():
    print("Missing slides")
    print(all_meta[all_meta["path"].isna()])
    
meta_stats = all_meta_with_tumor[~ (all_meta_with_tumor["path"].isna())]


In [ ]:
all_meta_with_tumor["tumor"].value_counts()

In [ ]:
stain_ct = pd.DataFrame(meta_stats.loc[:, "Stain"].value_counts())
stain_ct.columns=["Count"]
total = stain_ct["Count"].sum()
stain_ct["Perct"] = stain_ct["Count"].apply(lambda x: x / total * 100)
stain_ct.head(25)


In [ ]:
blocks = meta_stats.groupby(["UM Accession", "Block", "tumor"])["Stain"].agg(list).reset_index()
blocks

In [ ]:
fsx_blocks = blocks.apply(lambda x: x["Block"].endswith("x"), axis=1)
blocks[fsx_blocks]["tumor"].value_counts()

In [ ]:
heonly_blocks = blocks[~fsx_blocks].apply(lambda x: (len(set(x["Stain"]))==1) and (x["Stain"][0]=="H&E"), axis=1)
blocks[~fsx_blocks][heonly_blocks]["tumor"].value_counts()

In [ ]:
no_he = blocks[~fsx_blocks][~heonly_blocks]["Stain"].apply(lambda x: "H&E" not in x)
blocks[~fsx_blocks][~heonly_blocks][no_he]["tumor"].value_counts()

In [ ]:
def fs_in_all_blocks(x):
    return all(["FS" in xb for xb in x["Block"]]) 

per_blocks = blocks[~fsx_blocks][~heonly_blocks][~no_he]


num_fsx_blocks = fsx_blocks.sum()
num_heonly_blocks = heonly_blocks.sum()
num_nohe_blocks = no_he.sum()
num_rc_blocks = len(per_blocks)

In [ ]:
su_stains = blocks.groupby("UM Accession")["Stain"].agg(lambda x: list(chain(*x)))

In [ ]:
rc_sus = set(per_blocks["UM Accession"].tolist())
not_rc_sus = set(blocks[~ blocks["UM Accession"].isin(rc_sus)]["UM Accession"].tolist())

In [ ]:
num_rc_sus = len(rc_sus)
num_not_rc_sus = len(not_rc_sus)


In [ ]:
plt.rcParams.update({'font.size': 16})
fig, (ax1, ax2) = plt.subplots(1,2,figsize=(12, 6))
wedges, texts = ax1.pie([num_fsx_blocks, num_heonly_blocks, num_nohe_blocks, num_rc_blocks], labels=[f"Intraop FS only\n{num_fsx_blocks}", f"HE only\n{num_heonly_blocks}", f"HE missing\n{num_nohe_blocks}", f"Reg Cand\n{num_rc_blocks}"], colors=["lightgray","gray","pink", "green"], wedgeprops=dict(width=0.2), startangle=90)
ax1.set_title("Blocks")

wedges, texts = ax2.pie([num_not_rc_sus, num_rc_sus], labels=[f"Can't Reg\n{num_not_rc_sus}", f"Reg Cand\n{num_rc_sus}"], colors=["lightgray","green"], wedgeprops=dict(width=0.2), startangle=90)
ax2.set_title("SU cases")

plt.tight_layout()
matplotlib.rcParams['pdf.fonttype'] = 42 
#plt.savefig("data_quality.pdf")


In [ ]:
per_blocks = blocks[~fsx_blocks]

In [ ]:
all_stains = per_blocks["Stain"].explode().value_counts().reset_index()["Stain"].tolist()
all_stain_idx_map = {s:i for i,s in enumerate(all_stains)}

In [ ]:
#less_frequent_stains = per_blocks["Stain"].explode().value_counts()[64:].reset_index()["Stain"].tolist()
#top_stains = set(per_blocks["Stain"].explode().value_counts()[:64].reset_index()["Stain"].tolist())
#exploded_stains = per_blocks[["tumor", "Stain"]].explode("Stain")
#exploded_stains = exploded_stains[exploded_stains["Stain"].isin(top_stains)]
#stain_tumor_counts = (
#    exploded_stains.groupby(['Stain', 'tumor'])
#      .size()
#      .reset_index(name='stain_count')
#)
#assigned = (
#    stain_tumor_counts.sort_values('stain_count', ascending=False)
#                      .drop_duplicates(subset=['Stain'])
#)
#tumor_freq = exploded_stains['tumor'].value_counts().to_dict()
#assigned['tumor_freq'] = assigned['tumor'].map(tumor_freq)

# 4. Sort by overall tumor frequency (desc) then by stain frequency (desc)
#assigned_sorted = assigned.sort_values(['tumor_freq', 'stain_count'], ascending=False)

#all_stains = assigned_sorted["Stain"].tolist() + less_frequent_stains
#all_stain_idx_map = {s:i for i,s in enumerate(all_stains)}

In [ ]:
def get_dense_rep(x, all_stain_idx_map):
    curr_stains = x["Stain"]
    block_meta_fname = f"/nfs/turbo/umms-tocho-snr/exp/chengjia/mns_block_patch/{x['UM Accession']}.{x['Block']}_blockmeta.csv"
    feat = np.ones((len(all_stains), 3)) * [[244,252, 227]] / 255
    
    if os.path.exists(block_meta_fname):
        block_meta = pd.read_csv(block_meta_fname)
        
        if not "pixel_mean_r" in block_meta.columns:
            for cs in curr_stains:
                feat[all_stain_idx_map[cs],:] = np.array((0, 255, 0))/255
            return feat
        
        colors = block_meta.groupby("Stain")[["pixel_mean_r","pixel_mean_g","pixel_mean_b"]].agg("mean")

        for cs in set(curr_stains):
            if cs in colors.index:
                rgb = np.expand_dims(np.expand_dims(np.array(colors.loc[cs]),0),0).astype(np.uint8)
                hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
                hsv[:,:,1] =  np.minimum(hsv[:,:,1] * 16, 255)
                feat[all_stain_idx_map[cs],:] = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB).squeeze() / 255
            else:
                print(f"{cs} not patched in {x['UM Accession']}.{x['Block']}")
                feat[all_stain_idx_map[cs],:] = np.array((0, 255, 0))/255
    else:
        for cs in curr_stains:
            feat[all_stain_idx_map[cs],:] = np.array((0, 255, 0))/255
    return feat

In [ ]:
tumor_types = ["glioma", "circm_astro", "glioneuronal", "ependymal", "choroid", "embryonal", "pineal", "nerve", "mening", "mesenchymal", "chondro", "melanocytic", "hematolymphoid", "germ", "sellar", "met", "not_neo", "not_cns", "unk"]
tumor_type_idx = {c:i for i, c in enumerate(tumor_types)}

In [ ]:
import matplotlib

In [ ]:
per_blocks = blocks[~fsx_blocks]

In [ ]:
plt.rcParams.update({'font.size': 16})

per_blocks["stain_feat"] = per_blocks.apply(partial(get_dense_rep, all_stain_idx_map=all_stain_idx_map), axis=1)
#per_blocks = per_blocks[~(per_blocks["stain_feat"].isna())]

per_blocks["num_stains"] = per_blocks["Stain"].apply(lambda x: len(set(x)))
per_blocks["tumor_idx"] = [tumor_type_idx[tt] for tt in per_blocks["tumor"].tolist()]

prb_sorted = per_blocks.sort_values(["tumor_idx", "num_stains"])
stain_feat = np.array(prb_sorted["stain_feat"].tolist())

fig, (ax0, ax1) = plt.subplots(2,1,figsize=(25,15), gridspec_kw={'height_ratios': [1, 32]})

ims0 = ax0.imshow(np.expand_dims(np.array(prb_sorted["tumor_idx"]), axis=0), aspect="auto", interpolation='nearest', cmap="turbo")
#ax0.xaxis.set_major_locator(ticker.MultipleLocator(400))
ax0.set_yticks([0])
ax0.set_yticklabels(["tumor"])
ax0.tick_params(axis='both', which='both', length=0) 
ax0.set_xlabel('Blocks')
cmap = plt.get_cmap("turbo") 

legend_patches = [mpatches.Patch(color=cmap(i/(len(set(tumor_types))-1)), label=c) for i,c in enumerate(tumor_types)]
ax0.legend(handles=legend_patches, bbox_to_anchor=[0.5, 1.2], loc="lower center", ncols=7)


n_stains = 64

ax1.imshow(einops.rearrange(stain_feat, "b s c -> s b c")[:n_stains,...], aspect="auto",interpolation='nearest')
ax1.set_yticks(np.arange(n_stains))
ax1.set_yticklabels(all_stains[:n_stains])
ax1.tick_params(axis='both', which='both', length=0) 
ax1.set_xlabel('Blocks')
plt.tight_layout()
matplotlib.rcParams['pdf.fonttype'] = 42 
#plt.savefig("all_blocks.pdf")

In [ ]:
plt.rcParams.update({'font.size': 16})

per_blocks = blocks[~fsx_blocks][~heonly_blocks][~no_he]
per_blocks["stain_feat"] = per_blocks.apply(partial(get_dense_rep, all_stain_idx_map=all_stain_idx_map), axis=1)
#per_blocks = per_blocks[~(per_blocks["stain_feat"].isna())]

per_blocks["num_stains"] = per_blocks["Stain"].apply(lambda x: len(set(x)))
per_blocks["tumor_idx"] = [tumor_type_idx[tt] for tt in per_blocks["tumor"].tolist()]

prb_sorted = per_blocks.sort_values(["tumor_idx", "num_stains"])
stain_feat = np.array(prb_sorted["stain_feat"].tolist())

fig, (ax0, ax1) = plt.subplots(2,1,figsize=(25,15), gridspec_kw={'height_ratios': [1, 32]})

ims0 = ax0.imshow(np.expand_dims(np.array(prb_sorted["tumor_idx"]), axis=0), aspect="auto", interpolation='nearest', cmap="turbo")
#ax0.xaxis.set_major_locator(ticker.MultipleLocator(400))
ax0.set_yticks([0])
ax0.set_yticklabels(["tumor"])
ax0.tick_params(axis='both', which='both', length=0) 
ax0.set_xlabel('Blocks')
cmap = plt.get_cmap("turbo") 

legend_patches = [mpatches.Patch(color=cmap(i/(len(set(tumor_types))-1)), label=c) for i,c in enumerate(tumor_types)]
ax0.legend(handles=legend_patches, bbox_to_anchor=[0.5, 1.2], loc="lower center", ncols=7)


n_stains = 64

ax1.imshow(einops.rearrange(stain_feat, "b s c -> s b c")[:n_stains,...], aspect="auto",interpolation='nearest')
ax1.set_yticks(np.arange(n_stains))
ax1.set_yticklabels(all_stains[:n_stains])
ax1.tick_params(axis='both', which='both', length=0) 
ax1.set_xlabel('Blocks')
plt.tight_layout()
matplotlib.rcParams['pdf.fonttype'] = 42 
#plt.savefig("all_blocks.pdf")

In [ ]:
plt.rcParams.update({'font.size': 16})

su_gb = prb_sorted.groupby("UM Accession")
su_stains_feat = su_gb["stain_feat"].mean()
tumor_ = su_gb["tumor_idx"].first()
su_stain_count = pd.concat([su_stains_feat, tumor_], axis=1)
su_stain_count["num_stains"] = su_stain_count["stain_feat"].apply(sum)
su_stain_count = su_stain_count.sort_values(["tumor_idx"])
su_tain_feat = np.array(su_stain_count["stain_feat"].tolist())

fig, (ax0, ax1) = plt.subplots(2,1,figsize=(25,15), gridspec_kw={'height_ratios': [1, 32]})

ax0.imshow(np.expand_dims(su_stain_count["tumor_idx"].to_numpy(), 0), aspect="auto", interpolation='nearest', cmap="turbo")
#ax0.xaxis.set_major_locator(ticker.MultipleLocator(100))
ax0.set_yticks([0])
ax0.set_yticklabels(["tumor"])
ax0.tick_params(axis='both', which='both', length=0) 
ax0.set_xlabel('SU Numbers')

cmap = plt.get_cmap("turbo") 

legend_patches = [mpatches.Patch(color=cmap(i/(len(set(tumor_types))-1)), label=c) for i,c in enumerate(tumor_types)]
ax0.legend(handles=legend_patches, bbox_to_anchor=[0.5, 1.2], loc="lower center", ncols=7)

n_stains = 64

ax1.imshow(einops.rearrange(su_tain_feat, "b s c -> s b c")[:n_stains,...], aspect="auto",interpolation='nearest', cmap="Greens")
ax1.set_yticks(np.arange(n_stains))
ax1.set_yticklabels(all_stains[:n_stains])
ax1.tick_params(axis='both', which='both', length=0) 
ax1.set_xlabel('SU Numbers')
plt.tight_layout()
matplotlib.rcParams['pdf.fonttype'] = 42 
#plt.savefig("reg_cand.pdf")

```
plt.rcParams.update({'font.size': 16})

per_blocks["stain_feat"] = per_blocks.apply(partial(get_dense_rep, all_stain_idx_map=all_stain_idx_map), axis=1)
per_blocks["num_stains"] = per_blocks["Stain"].apply(lambda x: len(set(x)))
per_blocks["tumor_idx"] = [tumor_type_idx[tt] for tt in per_blocks["tumor"].tolist()]
prb_sorted = per_blocks.sort_values(["tumor_idx", "num_stains"])
stain_feat = np.array(prb_sorted["stain_feat"].tolist())

fig, (ax0, ax1) = plt.subplots(2,1,figsize=(20,15), gridspec_kw={'height_ratios': [1, 32]})

ims0 = ax0.imshow(np.expand_dims(np.array(prb_sorted["tumor_idx"]), axis=0), aspect="auto", interpolation='nearest', cmap="turbo")
#ax0.xaxis.set_major_locator(ticker.MultipleLocator(400))
ax0.set_yticks([0])
ax0.set_yticklabels(["tumor"])
ax0.tick_params(axis='both', which='both', length=0) 
ax0.set_xlabel('Blocks')
cmap = plt.get_cmap("turbo") 

legend_patches = [mpatches.Patch(color=cmap(i/(len(set(tumor_types))-1)), label=c) for i,c in enumerate(tumor_types)]
ax0.legend(handles=legend_patches, bbox_to_anchor=[0.5, 1.2], loc="lower center", ncols=7)

n_stains = 64

ax1.imshow(einops.rearrange(stain_feat, "b s c -> s b c")[:n_stains,...], aspect="auto",interpolation='nearest', cmap="Greens")
ax1.set_yticks(np.arange(n_stains))
ax1.set_yticklabels(all_stains[:n_stains])
ax1.tick_params(axis='both', which='both', length=0) 
ax1.set_xlabel('Blocks')
plt.tight_layout()
```

```
plt.rcParams.update({'font.size': 16})

su_gb = prb_sorted.groupby("UM Accession")
su_stains_feat = su_gb["stain_feat"].mean()
tumor_ = su_gb["tumor_idx"].first()
su_stain_count = pd.concat([su_stains_feat, tumor_], axis=1)
su_stain_count["num_stains"] = su_stain_count["stain_feat"].apply(sum)
su_stain_count = su_stain_count.sort_values(["tumor_idx"])
su_tain_feat = np.array(su_stain_count["stain_feat"].tolist())

fig, (ax0, ax1) = plt.subplots(2,1,figsize=(20,15), gridspec_kw={'height_ratios': [1, 32]})

ax0.imshow(np.expand_dims(su_stain_count["tumor_idx"].to_numpy(), 0), aspect="auto", interpolation='nearest', cmap="turbo")
#ax0.xaxis.set_major_locator(ticker.MultipleLocator(100))
ax0.set_yticks([0])
ax0.set_yticklabels(["tumor"])
ax0.tick_params(axis='both', which='both', length=0) 
ax0.set_xlabel('SU Numbers')

cmap = plt.get_cmap("turbo") 

legend_patches = [mpatches.Patch(color=cmap(i/(len(set(tumor_types))-1)), label=c) for i,c in enumerate(tumor_types)]
ax0.legend(handles=legend_patches, bbox_to_anchor=[0.5, 1.2], loc="lower center", ncols=7)

n_stains = 64

ax1.imshow(einops.rearrange(su_tain_feat, "b s c -> s b c")[:n_stains,...], aspect="auto",interpolation='nearest', cmap="Greens")
ax1.set_yticks(np.arange(n_stains))
ax1.set_yticklabels(all_stains[:n_stains])
ax1.tick_params(axis='both', which='both', length=0) 
ax1.set_xlabel('SU Numbers')
plt.tight_layout()
```


In [ ]:
all_ns = pd.read_csv("data/all_neuro_cases.csv")

In [ ]:
scanned_ns = pd.read_csv("data/scaned_neuro_cases.csv")

In [ ]:
all_ns = all_ns.merge(scanned_ns, on="class")
all_ns.loc[all_ns["class"]=="Diffuse Gliomas", "class"] = "glioma"
all_ns.loc[all_ns["class"]=="Sellar Region Tumors", "class"] = "sellar"
all_ns.loc[all_ns["class"]=="Meningiomas", "class"] = "mening"
all_ns.loc[all_ns["class"]=="Metastatic tumors", "class"] = "met"
all_ns.loc[all_ns["class"]=="Circumscribed Astrocytic Gliomas", "class"] = "circm_astro"
all_ns.loc[all_ns["class"]=="Glioneuronal and Neuronal Tumors", "class"] = "glioneuronal"
all_ns.loc[all_ns["class"]=="Non Neoplastic Brain Conditions", "class"] = "not_neo"
all_ns.loc[all_ns["class"]=="Embryonal Tumors", "class"] = "embryonal"
all_ns.loc[all_ns["class"]=="Cranial and Paraspinal Nerve Tumors", "class"] = "nerve"
all_ns.loc[all_ns["class"]=="Unknown", "class"] = "unk"
all_ns.loc[all_ns["class"]=="Mesenchymal Non Meningothelial Tumors", "class"] = "mesenchymal"
all_ns.loc[all_ns["class"]=="Ependymal Tumor", "class"] = "ependymal"
all_ns.loc[all_ns["class"]=="Non CNS Specimens", "class"] = "not_cns"
all_ns.loc[all_ns["class"]=="Hematolymphoid Tumors", "class"] = "hematolymphoid"
all_ns.loc[all_ns["class"]=="Chondro-osseous Tumors", "class"] = "chondro"
all_ns.loc[all_ns["class"]=="Pineal Tumors", "class"] = "pineal"
all_ns.loc[all_ns["class"]=="Choroid Plexus Tumor", "class"] = "choroid"
all_ns.loc[all_ns["class"]=="Germ Cell Tumors", "class"] = "germ"
all_ns.loc[all_ns["class"]=="Melanocytic Tumors", "class"] = "melanocytic"



In [ ]:
all_ns

In [ ]:
all_ns[::-1]

In [ ]:
all_ns["all_txt"] = all_ns.apply(lambda x: f"{x['class']} {x['all_count']}", axis=1)
all_ns["scanned_txt"] = all_ns.apply(lambda x: f"{x['class']} {x['scanned_count']}", axis=1)

In [ ]:
plt.rcParams.update({'font.size': 20})
fig, ax = plt.subplots(1,1,figsize=(16, 8))
ax.barh(y=range(len(all_ns)), width=all_ns["all_count"][::-1], color="#f76707", zorder=2)
ax.barh(y=range(len(all_ns)), width=all_ns["scanned_count"][::-1], color="#74b816",zorder=2)
ax.set_yticks(range(len(all_ns)))
ax.set_yticklabels(all_ns["class"][::-1])
ax.xaxis.grid()
ax.tick_params('both', length=0)
for i, r in all_ns[::-1].reset_index().iterrows():
    #print(i)
    #print(r["all_count"])
    if r["all_count"]>1500:
        ax.text(r["all_count"]-10, i, f"{r['scanned_count']} / {r['all_count']}", horizontalalignment='right', verticalalignment='center', color="white")
        
    else:
        ax.text(r["all_count"]+10, i, f"{r['scanned_count']} / {r['all_count']}", horizontalalignment='left', verticalalignment='center')
ax.set_title("Estimated neuro SU cases (1997 - Present)\nTotal: 1678 / 14063 scanned")
plt.tight_layout()
matplotlib.rcParams['pdf.fonttype'] = 42 
plt.savefig("all_case_report_bar.pdf")

In [ ]:
plt.rcParams.update({'font.size': 16})
fig, (ax1, ax2) = plt.subplots(1,2,figsize=(16, 8))
wedges, texts = ax1.pie(all_ns["all_count"], labels=all_ns["all_txt"], wedgeprops=dict(width=0.2), startangle=0)
ax2.set_title("Scanned Neuro SU cases\nTotal=1678")

wedges, texts = ax2.pie(all_ns["scanned_count"], labels=all_ns["scanned_txt"], wedgeprops=dict(width=0.2), startangle=0)
ax1.set_title("Estimated Neuro SU cases\nTotal=14063")

plt.tight_layout()
matplotlib.rcParams['pdf.fonttype'] = 42 
plt.savefig("all_case_report.pdf")

In [ ]:

wedges, texts = ax2.pie([num_not_rc_sus, num_rc_sus], labels=[f"Can't Reg\n{num_not_rc_sus}", f"Reg Cand\n{num_rc_sus}"], colors=["lightgray","green"], wedgeprops=dict(width=0.2), startangle=90)
ax2.set_title("all SU cases")

plt.tight_layout()
matplotlib.rcParams['pdf.fonttype'] = 42 
plt.savefig("data_quality.pdf")

In [ ]:
not_fsx = all_meta[~all_meta["Block"].str.endswith("x")]
len(not_fsx[not_fsx["Stain"]=="H&E"])

In [ ]:
len(not_fsx[not_fsx["Stain"]!="H&E"])